In [ ]:
#SystemMessage dinamico, e ROUTING DYNAMICO (builder.add_conditional_edges("analyze", route_by_sentiment, ["positive_response", "negative_response"])  #🔥🔥CONDITIONAL EDGE! in base al valore di ritorno di route_by_sentiment (che è una stringa) decido quale nodo eseguire ('positive response' oppure 'negative response') dopo node 'analyze')

In [ ]:
from typing_extensions import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field


# Configuration
BASE_URL = "http://localhost:11434"
MODEL_NAME = "deepseek-r1"  # Ensure this model is pulled in Ollama

llm = ChatOllama(
    model= MODEL_NAME, 
    base_url=BASE_URL)

In [ ]:

class SentimentAnalysis(BaseModel):  #schema structure for the output of the sentiment analysis!
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment classification either positive or negative")
    confidence: float = Field(ge=0, le=1.0, description="Confidence score from 0.0 to 1.0")  #ge=greater than or equal to, le=less than or equal to. setti le contraints per il campo confidence
    reason: str = Field(description="Brief explanation")


class SentimentState(TypedDict):  #stato (che verra condiviso tra tutti i nodes)
    original_tweet: str
    sentiment: str
    confidence: float
    response_tweet: str

In [ ]:
def analyze_sentiment(state: SentimentState):  #nodo langgraph per analizzare il sentiment del tweet del cliente
    tweet = state['original_tweet']
    print(f"analyzing customer tweet: {tweet}")

    structured_llm = llm.with_structured_output(SentimentAnalysis)  #istanza llm con output strutturato custom! ora il mdello provera a produrre SentimentAnalysis(...) non un testo libero
    # ALT +Z
    messages = [
        SystemMessage("Analyze sentiment and provide the structured output. Use 0 to 1.0 scale for confidence. lower is negative and higher is positive"),
        HumanMessage(tweet)
    ]
    analysis = structured_llm.invoke(messages)
    print(f"Sentiment Analysis is done:\n{analysis}")
    return {'sentiment': analysis.sentiment,
            'confidence': analysis.confidence}  #return stato update (non necessario tuuutti i campi)

state = {'original_tweet': "Just launched my new product!"}  #stato iniziale custom
analyze_sentiment(state)  #go!

In [ ]:

#routing condizionale, DECISION GRAPH

def generate_postive_response(state: SentimentState):  #nodo langgraph
    print(f"current state in positive respone node: {state}")
    messages = [
        SystemMessage(f"""Generate a warm response to this positive tweet under 280 chars.
                      Confidence: {state['confidence']}. High confidence means be enthusiastic otherwise be friendly."""),
        HumanMessage(state['original_tweet'])
    ]  #il SystemMessage è 🔥DINAMICO! infatti piu è alto il state['confidence'] piu il settagio del SystemMessage è diverso!!!
    response = llm.invoke(messages)
    return {'response_tweet': response.content.strip()}  #return stato update (non necessario tuuutti i campi), senza spaced a sx e dx


def generate_negative_response(state: SentimentState):
    print(f"current state in negative respone node: {state}")
    messages = [
        SystemMessage(f"""Generate and empathetic response to this negative tweet under 280 chars.
                      If Confidence {state['confidence']} is very low then be empathetic otherwise 
                      be understanding."""),
        HumanMessage(state['original_tweet'])
    ]   #il SystemMessage è 🔥DINAMICO! infatti piu è alto il state['confidence'] piu il settagio del SystemMessage è diverso!!!
    response = llm.invoke(messages)
    return {'response_tweet': response.content.strip()}


def route_by_sentiment(state: SentimentState):
    if state['sentiment'] == 'positive':
        return "positive_response"
    else:
        return "negative_response"
    

def create_router_graph():
    builder = StateGraph(SentimentState)
    # add nodes
    builder.add_node("analyze", analyze_sentiment)
    builder.add_node("positive_response", generate_postive_response)
    builder.add_node("negative_response", generate_negative_response)
    #create edges
    builder.add_edge(START, "analyze")
    builder.add_conditional_edges("analyze", route_by_sentiment, ["positive_response", "negative_response"])  #🔥🔥CONDITIONAL EDGE! in base al valore di ritorno di route_by_sentiment (che è una stringa) decido quale nodo eseguire ('positive response' oppure 'negative response') dopo node 'analyze'
    builder.add_edge("positive_response", END)
    builder.add_edge("negative_response", END)

    graph = builder.compile()
    return graph

graph = create_router_graph()  #save in var
graph

In [ ]:
tweet = "Just launched my new product! the response from everyone has been amazing so far."

result = graph.invoke({'original_tweet': tweet})  #passi stato iniale del grafo, e runni grafo
result

In [ ]:
tweet = "Really disappointed with the service I received today."

result = graph.invoke({'original_tweet': tweet})
result